In [ ]:
import torch
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image
from scipy.io import loadmat
import torch.backends.cudnn as cudnn
import os
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from model import PINet_cpx_v6
from model import PINet_cpx_v6_pre
import torch.nn.functional as F
from mydataset import ComplexDataset
from myloss import ComplexLoss_amp_phs,ComplexLoss_re_im
from utilities import *
   

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data_folder = r'./pinet_dataset'
val_data_folder = os.path.join(data_folder,'val')
test_data_folder = os.path.join(data_folder,'test')
val_data_folder_amp = os.path.join(val_data_folder,'amp')
val_data_folder_phs = os.path.join(val_data_folder,'phs')
test_data_folder_amp = os.path.join(test_data_folder,'amp')
test_data_folder_phs = os.path.join(test_data_folder,'phs')


model_list = []
model_path_list = []
model_folder = 'model_saved_pinet_compared3'
# model_folder = 'model_saved_pinet_compared2'
size = 256
fold_iters = 9



transform = transforms.Compose([
    transforms.Resize((size, size)),  # 调整大小
    transforms.ToTensor()
])

def load_checkpoint(checkpoint_path, model):
    if os.path.isfile(checkpoint_path):
        # print(f"Loading checkpoint from {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        # print(f"Checkpoint loaded, starting from epoch {epoch + 1}")
    else:
        print("No checkpoint found, starting from scratch.")

## HSTA

In [ ]:
from scipy.io import loadmat
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim


data_JZX = loadmat('real_data/JZX1.mat')
for s in data_JZX.keys():
    if not s.startswith('__'):
        print(f'data_JZX[\'{s}\'].shape:{data_JZX[s].shape}')
        print(f'data_JZX[\'{s}\'].dtype:{data_JZX[s].dtype}')
        
percent = 0.5
label_JZX = data_JZX['O_global']
label_JZX_amp = np.abs(label_JZX)
label_JZX_phs = np.angle(label_JZX)
vmin_label_phs = np.percentile(label_JZX_phs, percent)
vmax_label_phs = np.percentile(label_JZX_phs, 100-percent)
vmin_label_amp = np.percentile(label_JZX_amp, percent)
vmax_label_amp = np.percentile(label_JZX_amp, 100-percent)
print(f'range of JZX amp: {vmin_label_amp:.3f} to {vmax_label_amp:.3f}',)
print(f'range of JZX phs: {vmin_label_phs:.3f} to {vmax_label_phs:.3f}',)


label_data = loadmat('real_data/Eout_HSTA.mat')
label = label_data['Eout'][0:Nx,0:Ny]
label_amp = np.abs(label)
label_phs = np.angle(label)
vmin_label_phs = np.percentile(label_phs, percent)
vmax_label_phs = np.percentile(label_phs, 100-percent)
vmin_label_amp = np.percentile(label_amp, percent)
vmax_label_amp = np.percentile(label_amp, 100-percent)
print(f'range of amp: {vmin_label_amp:.3f} to {vmax_label_amp:.3f}')
print(f'range of phs: {vmin_label_phs:.3f} to {vmax_label_phs:.3f}')
label_amp = np.clip(label_amp,vmin_label_amp,vmax_label_amp)
label_phs = np.clip(label_phs,vmin_label_phs,vmax_label_phs)
label_amp = (label_amp - label_amp.min()) / (label_amp.max() - label_amp.min())
label_phs = (label_phs - label_phs.min()) / (label_phs.max() - label_phs.min())

plt.figure()
plt.imshow(label_amp, cmap='gray')
plt.title(f'Amplitude')
plt.axis('off')
plt.show()
plt.figure()
plt.imshow(label_phs, cmap='gray')
plt.title(f'Phase')
plt.axis('off')
plt.show()
# 展平成一维方便画图
vals = label_amp.ravel()
plt.figure()
plt.hist(vals, bins=100)   # range=(vmin_amp,vmax_amp)
plt.ylabel("count")
plt.title("Histogram of label amplitude")
plt.show()
# 展平成一维方便画图
vals = label_phs.ravel()
plt.figure()
plt.hist(vals, bins=100)   # bin 数可以自己调
plt.xlabel("phase (rad)")
plt.ylabel("count")
plt.title("Histogram of label phase")
plt.show()
# plt.imsave(f'model_saved_pinet_v29/phs_result_HSTA_label.png',label_phs,cmap='gray')

data = loadmat('real_data/HSTA.mat')
print(data.keys())
distance = data['dist']
shift_samples = data['Shift_Samples']
shift_samples_0 = data['Shift_Samples_0']
pixelsize = data['pixelsize']
print(distance)
print(shift_samples.shape)
print(shift_samples_0.shape)
print(pixelsize)
print(data['Lambda'])

diff_np = np.sqrt(shift_samples[:,:,0])
diff_np = diff_np[0:Nx,0:Ny]
diff_np = ( diff_np - diff_np.min() ) / ( diff_np.max() - diff_np.min() )
plt.imshow(diff_np,cmap='gray')
plt.title('diff_data')
plt.axis('off')
plt.show()


diff_tensor = torch.from_numpy(diff_np).cuda().float()
print(torch.max(diff_tensor))
print(torch.min(diff_tensor))
print(diff_tensor.shape,diff_tensor.dtype)



with torch.no_grad():
    for i in range(15,21):
        num = i*10
        model = PINet_cpx_v6(fold_iters=fold_iters).to(device)
        model_path = f'model_pinet_size256_epoch_{num}_batchsize4_2mm_to_3mm.pth'
        checkpoint_path = os.path.join(model_folder,model_path)
        load_checkpoint(checkpoint_path, model)
        
        pred,y_rec = model(diff_tensor.unsqueeze(0).unsqueeze(0),TF_torch)
        pred_np = pred.squeeze(0).squeeze(0).detach().cpu().numpy()
        pred_amp_np = np.abs(pred_np)
        pred_phs_np = np.angle(pred_np)

        vmin_phs = np.percentile(pred_phs_np, percent)
        vmax_phs = np.percentile(pred_phs_np, 100-percent)
        vmin_amp = np.percentile(pred_amp_np, percent)
        vmax_amp = np.percentile(pred_amp_np, 100-percent)
        pred_amp_np = np.clip(pred_amp_np,vmin_amp,vmax_amp)
        pred_phs_np = np.clip(pred_phs_np,vmin_phs,vmax_phs)
        
        pred_amp_np = (pred_amp_np - pred_amp_np.min()) / (pred_amp_np.max() - pred_amp_np.min())
        pred_phs_np = (pred_phs_np - pred_phs_np.min()) / (pred_phs_np.max() - pred_phs_np.min())
        # pred_amp_np = pred_amp_np ** 2
        # pred_phs_np = np.sqrt(pred_phs_np)
        
        print(f'range of amp: {vmin_amp:.3f} to {vmax_amp:.3f}',)
        print(f'range of phs: {vmin_phs:.3f} to {vmax_phs:.3f}',)

        psnr_amp = psnr(label_amp,pred_amp_np, data_range=1.0)
        psnr_phs = psnr(label_phs,pred_phs_np, data_range=1.0)
        ssim_amp = ssim(label_amp,pred_amp_np, data_range=1.0)
        ssim_phs = ssim(label_phs,pred_phs_np, data_range=1.0)        

        print(f'psnr of amp: {psnr_amp:.3f}dB')
        print(f'psnr of phs: {psnr_phs:.3f}dB')        
        print(f'ssim of amp: {ssim_amp:.3f}')
        print(f'ssim of phs: {ssim_phs:.3f}')  
        
        plt.figure()
        plt.imshow(pred_amp_np, cmap='gray')
        plt.title(f'Amplitude {num} epochs')
        plt.axis('off')
        plt.show()
        
        
        plt.figure()
        plt.imshow(pred_phs_np, cmap='gray')
        plt.title(f'Phase {num} epochs')
        plt.axis('off')
        plt.show()
             
        # 展平成一维方便画图
        vals = pred_amp_np.ravel()
        
        plt.figure()
        plt.hist(vals, bins=100)   # range=(vmin_amp,vmax_amp)
        plt.ylabel("count")
        plt.title("Histogram of predicted amplitude")
        plt.show()
        
        
        # 展平成一维方便画图
        vals = pred_phs_np.ravel()
        
        plt.figure()
        plt.hist(vals, bins=100)   # bin 数可以自己调
        plt.xlabel("phase (rad)")
        plt.ylabel("count")
        plt.title("Histogram of predicted phase")
        plt.show()
        # # plt.imsave(f'diff_Beewings.png',diff_np,cmap='gray')
        plt.imsave(os.path.join(model_folder,'amp_result_HSTA_ours.png'),pred_amp_np,cmap='gray')
        plt.imsave(os.path.join(model_folder,'phs_result_HSTA_ours.png'),pred_phs_np,cmap='gray')

In [ ]:
# # plt.imsave(f'diff_Beewings.png',diff_np,cmap='gray')
# plt.imsave(f'model_saved_pinet_v29/amp_result_HSTA_ours.png',pred_amp_np,cmap='gray')
# plt.imsave(f'model_saved_pinet_v29/phs_result_HSTA_ours.png',pred_phs_np,cmap='gray')

In [ ]:
# from scipy.io import loadmat
# import matplotlib.pyplot as plt
# import numpy as np

# data = loadmat('real_data/HSTA.mat')

# print(data.keys())
# distance = data['dist']
# shift_samples = data['Shift_Samples_0']
# print(distance)
# print(shift_samples.shape)

# diff_np = np.sqrt(shift_samples[:,:,0])
# diff_np = diff_np[0:Nx,0:Ny]
# plt.imshow(diff_np,cmap='gray')
# plt.title('diff_data')
# plt.axis('off')
# plt.show()

# diff_tensor = torch.from_numpy(diff_np).cuda().float()
# print(torch.max(diff_tensor))
# print(torch.min(diff_tensor))
# print(diff_tensor.shape,diff_tensor.dtype)

# with torch.no_grad():
#     for i in range(14,15):
#         model = PINet_cpx_v6(fold_iters=fold_iters).to(device)
#         model_path = f'model_pinet_size256_epoch_140_batchsize8_2.744mm.pth'
#         checkpoint_path = os.path.join(model_folder,model_path)
#         load_checkpoint(checkpoint_path, model)
        
        
    
#         pred,y_rec = model(diff_tensor.unsqueeze(0).unsqueeze(0),TF_torch)
#         print(pred.shape,pred.dtype)
#         print(torch.max(torch.angle(pred)))
#         print(torch.min(torch.angle(pred)))
        
        
        
        
#         pred_np = pred.squeeze(0).squeeze(0).detach().cpu().numpy()
#         pred_amp_np = np.abs(pred_np)
#         pred_phs_np = np.angle(pred_np)
        
#         pred_phs_np = np.clip(pred_phs_np,-0.4,0.3)
        
#         plt.imshow(pred_amp_np, cmap='gray')
#         plt.title('Amplitude')
#         # plt.colorbar()
#         plt.axis('off')
#         plt.show()
        
#         plt.imshow(pred_phs_np, cmap='gray')
#         plt.title(f'Phase {i}0 epochs')
#         # plt.colorbar()
#         plt.axis('off')
#         plt.show()
             
#         # 展平成一维方便画图
#         vals = pred_phs_np.ravel()
        
#         plt.figure()
#         plt.hist(vals, bins=100)   # bin 数可以自己调
#         plt.xlabel("phase (rad)")
#         plt.ylabel("count")
#         plt.title("Histogram of predicted phase")
#         plt.show()
        
        
#         # 展平成一维方便画图
#         vals = pred_amp_np.ravel()
        
#         plt.figure()
#         plt.hist(vals, bins=100)   # bin 数可以自己调
#         plt.ylabel("count")
#         plt.title("Histogram of predicted amplitude")
#         plt.show()
# plt.imsave(f'diff_HSTA.png',diff_np,cmap='gray')
# plt.imsave(f'amp_result_HSTA.png',pred_amp_np,cmap='gray')
# plt.imsave(f'phs_result_HSTA.png',pred_phs_np,cmap='gray')

## JZX

In [ ]:
# data = loadmat('real_data/JZX1.mat')

# print(data.keys())

# data_O_global = data['O_global']
# print(data_O_global.shape,data_O_global.dtype)



# plt.imshow(np.abs(data_O_global), cmap='gray')
# plt.title('O_global')
# plt.axis('off')
# plt.show()

# plt.imshow(np.angle(data_O_global), cmap='gray',vmax=2,vmin=-2)
# plt.title('O_global')
# plt.axis('off')
# plt.show()
# diff = data['images']


# # 展平成一维方便画图
# vals = np.angle(data_O_global).ravel()

# plt.figure()
# plt.hist(vals, bins=100)   # bin 数可以自己调
# plt.xlabel("0_global phase (rad)")
# plt.ylabel("count")
# plt.title("Histogram of predicted phase")
# plt.show()


# # 展平成一维方便画图
# vals = np.abs(data_O_global).ravel()

# plt.figure()
# plt.hist(vals, bins=100)   # bin 数可以自己调
# plt.xlabel("0_global phase (rad)")
# plt.ylabel("count")
# plt.title("Histogram of predicted amplitude")
# plt.show()



# diff = data['images']


# distance = [1.5025659,1.55608408,1.65219572,1.74330953,1.80327773]
    
# # z = np.array([distance[0] * 1e-3]) # Propagation distance (in meters)
# # # z = np.array([1.5 * 1e-3]) # Propagation distance (in meters)
# # TF = np.exp(1j*z*np.sqrt(k**2-(2*np.pi*FX)**2-(2*np.pi*FY)**2))
# # TF_torch = torch.from_numpy(TF).to(device).to(torch.complex64)
# diff_np = np.sqrt(diff[:,:,0])
# # diff_np = diff[:,:,0]
# print(diff_np.shape)
# diff_np = diff_np[0:Nx,0:Ny]
# print(diff_np.shape)

# diff_tensor = torch.from_numpy(diff_np).cuda().float()
# print(torch.max(diff_tensor))
# print(torch.min(diff_tensor))
# print(diff_tensor.shape,diff_tensor.dtype)

# with torch.no_grad():
#     for i in range(3,4):
#         model = PINet_cpx_v6(fold_iters=fold_iters).to(device)
#         model_path = f'model_pinet_size256_epoch_40_batchsize4_1.5mm_to_2mm.pth'
#         checkpoint_path = os.path.join(model_folder,model_path)
#         load_checkpoint(checkpoint_path, model)
        
        
    
#         pred,y_rec = model(diff_tensor.unsqueeze(0).unsqueeze(0),TF_torch)
#         print(pred.shape,pred.dtype)
#         print(torch.max(torch.angle(pred)))
#         print(torch.min(torch.angle(pred)))
        
        
        
        
#         pred_np = pred.squeeze(0).squeeze(0).detach().cpu().numpy()
#         pred_amp_np = np.abs(pred_np)
#         pred_phs_np = np.angle(pred_np)
        
#         pred_phs_np = np.clip(pred_phs_np,-1,1)
        
#         plt.imshow(pred_amp_np, cmap='gray')
#         plt.title('Amplitude')
#         # plt.colorbar()
#         plt.axis('off')
#         plt.show()
        
#         plt.imshow(pred_phs_np, cmap='gray')
#         plt.title(f'Phase {i}0 epochs')
#         # plt.colorbar()
#         plt.axis('off')
#         plt.show()
             
#         # 展平成一维方便画图
#         vals = pred_phs_np.ravel()
        
#         plt.figure()
#         plt.hist(vals, bins=100)   # bin 数可以自己调
#         plt.xlabel("phase (rad)")
#         plt.ylabel("count")
#         plt.title("Histogram of predicted phase")
#         plt.show()
        
        
#         # 展平成一维方便画图
#         vals = pred_amp_np.ravel()
        
#         plt.figure()
#         plt.hist(vals, bins=100)   # bin 数可以自己调
#         plt.ylabel("count")
#         plt.title("Histogram of predicted amplitude")
#         plt.show()

# plt.imsave(f'amp_result_JZX.png',pred_amp_np,cmap='gray')
# plt.imsave(f'phs_result_JZX.png',pred_phs_np,cmap='gray')


In [ ]:
from scipy.io import loadmat
import matplotlib.pyplot as plt
import numpy as np

## USAF 纯振幅

In [ ]:
# usaf = loadmat('real_data/USAF_20231107.mat')
# print(usaf.keys())

In [ ]:
# print(usaf['Shift_Samples'].shape)
# print(usaf['Shift_Samples_0'].shape)
# print(usaf['dist'].shape)
# print(usaf['Lambda'].shape)
# print(usaf['pixelsize'].shape)

# print(usaf['dist'])
# print(usaf['pixelsize'])


In [ ]:
# shift_samples = usaf['Shift_Samples']
# shift_samples_0 = usaf['Shift_Samples_0']
# dist = usaf['dist'][0]
# Lambda = usaf['Lambda'][0,0]
# pixelsize = usaf['pixelsize'][0,0]


# diff_np = np.sqrt(shift_samples_0[:,:,1])
# diff_np = diff_np[0:Nx,0:Ny]
# plt.imshow(diff_np,cmap='gray')
# plt.title('diff_data')
# plt.axis('off')
# plt.show()

# diff_tensor = torch.from_numpy(diff_np).cuda().float()
# print(torch.max(diff_tensor))
# print(torch.min(diff_tensor))
# print(diff_tensor.shape,diff_tensor.dtype)

# with torch.no_grad():
#     for i in range(14,15):
#         model = PINet_cpx_v6(fold_iters=fold_iters).to(device)
#         model_path = f'model_pinet_size256_epoch_40_batchsize4_1.5mm_to_2mm.pth'
#         checkpoint_path = os.path.join(model_folder,model_path)
#         load_checkpoint(checkpoint_path, model)
        
        
    
#         pred,y_rec = model(diff_tensor.unsqueeze(0).unsqueeze(0),TF_torch)
#         print(pred.shape,pred.dtype)
#         print(torch.max(torch.angle(pred)))
#         print(torch.min(torch.angle(pred)))
        
        
        
        
#         pred_np = pred.squeeze(0).squeeze(0).detach().cpu().numpy()
#         pred_amp_np = np.abs(pred_np)
#         pred_phs_np = np.angle(pred_np)
        
#         pred_phs_np = np.clip(pred_phs_np,-1,1)
        
#         plt.imshow(pred_amp_np, cmap='gray')
#         plt.title('Amplitude')
#         # plt.colorbar()
#         plt.axis('off')
#         plt.show()
        
#         plt.imshow(pred_phs_np, cmap='gray')
#         plt.title(f'Phase {i}0 epochs')
#         # plt.colorbar()
#         plt.axis('off')
#         plt.show()
             
#         # 展平成一维方便画图
#         vals = pred_phs_np.ravel()
        
#         plt.figure()
#         plt.hist(vals, bins=100)   # bin 数可以自己调
#         plt.xlabel("phase (rad)")
#         plt.ylabel("count")
#         plt.title("Histogram of predicted phase")
#         plt.show()
        
        
#         # 展平成一维方便画图
#         vals = pred_amp_np.ravel()
        
#         plt.figure()
#         plt.hist(vals, bins=100)   # bin 数可以自己调
#         plt.ylabel("count")
#         plt.title("Histogram of predicted amplitude")
#         plt.show()
        
# plt.imsave(f'diff_USAF.png',diff_np,cmap='gray')
# plt.imsave(f'amp_result_USAF.png',pred_amp_np,cmap='gray')
# plt.imsave(f'phs_result_USAF.png',pred_phs_np,cmap='gray')

In [ ]:
# for i in range(1):
#     plt.figure()
#     plt.imshow(shift_samples_0[:,:,1], cmap='gray')
#     plt.title(f'Shift Sample 0 - Image {i}')
#     plt.axis('off')
#     plt.show()
#     plt.imshow(shift_samples[:,:,i], cmap='gray')
#     plt.title(f'Shift Sample - Image {i}')
#     plt.axis('off')
#     plt.show()
    

In [ ]:

model = PINet_cpx_v6(fold_iters=fold_iters).to(device)
model_name = f'model_pinet_size256_epoch_130_batchsize8.pth'
checkpoint_path = os.path.join(model_folder,model_name)
load_checkpoint(checkpoint_path, model)

# for i in range(8):

    
#     diff_np = shift_samples_0[:,:,i]/255.0
#     diff_np = np.sqrt(diff_np)
#     print(diff_np.shape)
#     diff_tensor = torch.from_numpy(diff_np).cuda().float()
    
#     print(torch.max(diff_tensor))
#     print(torch.min(diff_tensor))
#     print(diff_tensor.shape,diff_tensor.dtype)
    
    
    
#     z = dist[i] * 1e-3 # Propagation distance (in meters)
    
#     TF = np.exp(1j*z*np.sqrt(k**2-(2*np.pi*FX)**2-(2*np.pi*FY)**2))
#     TF_torch = torch.from_numpy(TF).to(device).to(torch.complex64)

#     pred,y_rec = model(diff_tensor.unsqueeze(0).unsqueeze(0),TF_torch)
#     print(pred.shape,pred.dtype)
#     print(torch.max(torch.angle(pred)))
#     print(torch.min(torch.angle(pred)))
#     pred_np = pred.squeeze(0).squeeze(0).detach().cpu().numpy()
#     pred_amp_np = np.abs(pred_np)
#     pred_phs_np = np.angle(pred_np)
    
#     plt.imshow(diff_np, cmap='gray')
#     plt.title(f'Input Diffraction: {dist[i]}mm')
#     plt.axis('off')
#     plt.show()
    
#     plt.imshow(pred_amp_np, cmap='gray')
#     plt.title('Amplitude')
#     # plt.colorbar()
#     plt.axis('off')
#     plt.show()
#     # plt.imshow(np.clip(pred_phs_np,-3.6,3), cmap='gray')
#     plt.imshow(pred_phs_np, cmap='gray')
#     plt.title(f'Phase')
#     # plt.colorbar()
#     plt.axis('off')
#     plt.show()


# 纯相位

In [ ]:
# mxwky = loadmat('MXWKY1_20240425.mat')

In [ ]:
# print(mxwky.keys())
# print(mxwky['Shift_Samples'].shape)
# print(mxwky['Shift_Samples_0'].shape)
# print(mxwky['Auto_Dis'])
# print(mxwky['diss'])

In [ ]:
# # print(data.keys())
# # shift_samples = data['Shift_Samples']

# diff_np = np.sqrt(mxwky['Shift_Samples_0'][:,:,0])
# diff_np = diff_np[0:Nx,0:Ny]
# plt.imshow(diff_np,cmap='gray')
# plt.title('diff_data')
# plt.axis('off')
# plt.show()

# diff_tensor = torch.from_numpy(diff_np).cuda().float()
# print(torch.max(diff_tensor))
# print(torch.min(diff_tensor))
# print(diff_tensor.shape,diff_tensor.dtype)

# with torch.no_grad():
#     for i in range(4,5):
#         model = PINet_cpx_v6(fold_iters=fold_iters).to(device)
#         model_path = f'model_pinet_size256_epoch_{i}0_batchsize4_1.5mm_to_2mm.pth'
#         checkpoint_path = os.path.join(model_folder,model_path)
#         load_checkpoint(checkpoint_path, model)
        
        
    
#         pred,y_rec = model(diff_tensor.unsqueeze(0).unsqueeze(0),TF_torch)
#         print(pred.shape,pred.dtype)
#         print(torch.max(torch.angle(pred)))
#         print(torch.min(torch.angle(pred)))
        
        
        
        
#         pred_np = pred.squeeze(0).squeeze(0).detach().cpu().numpy()
#         pred_amp_np = np.abs(pred_np)
#         pred_phs_np = np.angle(pred_np)
        
#         pred_phs_np = np.clip(pred_phs_np,-1,1)
        
#         plt.imshow(pred_amp_np, cmap='gray')
#         plt.title('Amplitude')
#         # plt.colorbar()
#         plt.axis('off')
#         plt.show()
        
#         plt.imshow(pred_phs_np, cmap='gray')
#         plt.title(f'Phase {i}0 epochs')
#         # plt.colorbar()
#         plt.axis('off')
#         plt.show()
             
#         # 展平成一维方便画图
#         vals = pred_phs_np.ravel()
        
#         plt.figure()
#         plt.hist(vals, bins=100)   # bin 数可以自己调
#         plt.xlabel("phase (rad)")
#         plt.ylabel("count")
#         plt.title("Histogram of predicted phase")
#         plt.show()
        
        
#         # 展平成一维方便画图
#         vals = pred_amp_np.ravel()
        
#         plt.figure()
#         plt.hist(vals, bins=100)   # bin 数可以自己调
#         plt.ylabel("count")
#         plt.title("Histogram of predicted amplitude")
#         plt.show()
# plt.imsave(f'diff_mxwky.png',diff_np,cmap='gray')
# plt.imsave(f'amp_result_mxwky.png',pred_amp_np,cmap='gray')
# plt.imsave(f'phs_result_mxwky.png',pred_phs_np,cmap='gray')
    

## 整个测试集测试

In [ ]:
# import time
# psnr_amp_list = []
# psnr_phs_list = []
# ssim_amp_list = []
# ssim_phs_list = []
# time_list = []
# count = 0

# for img_diff_tensor, label_tensor in dataset_val:
#     count += 1
#     label_tensor = label_tensor.to(device)
#     start_time = time.time()
#     pred, y_rec = model(img_diff_tensor.unsqueeze(0),TF_torch)
#     pred = pred.squeeze(0)
#     end_time = time.time()
#     time_used = end_time - start_time
#     time_list.append(time_used)
#     psnr_amp = compute_psnr_skimage_single(torch.abs(pred),torch.abs(label_tensor))
#     psnr_phs = compute_psnr_skimage_single(torch.angle(pred),torch.angle(label_tensor))
#     ssim_amp = compute_ssim_skimage_single(torch.abs(pred),torch.abs(label_tensor))
#     ssim_phs = compute_ssim_skimage_single(torch.angle(pred),torch.angle(label_tensor))
#     psnr_amp_list.append(psnr_amp)
#     psnr_phs_list.append(psnr_phs)
#     ssim_amp_list.append(ssim_amp)
#     ssim_phs_list.append(ssim_phs)
#     print(f'image {count}: psnr_amp: psnr_amp:{psnr_amp:.2f}dB,psnr_phs:{psnr_phs:.2f}dB,ssim_amp:{ssim_amp:.2f},ssim_phs:{ssim_phs:.2f}, time:{time_used:.2f}s')

# psnr_amp_avg = np.mean(psnr_amp_list)
# psnr_phs_avg = np.mean(psnr_phs_list)
# ssim_amp_avg = np.mean(ssim_amp_list)
# ssim_phs_avg = np.mean(ssim_phs_list)
# time_avg = np.mean(time_list)
# print(f"{model_path}: psnr_amp:{psnr_amp_avg:.2f}dB,psnr_phs:{psnr_phs_avg:.2f}dB,ssim_amp:{ssim_amp_avg:.2f},ssim_phs:{ssim_phs_avg:.2f},time_avg:{time_avg:.2f}s")  

In [ ]:
# idx_list = []
# for i in range(200):
#     if psnr_phs_list[i] >= 20 and ssim_phs_list[i] >= 0.7:
#         print(f'image {i+1}: psnr_amp: psnr_amp:{psnr_amp_list[i]:.2f}dB,psnr_phs:{psnr_phs_list[i]:.2f}dB,ssim_amp:{ssim_amp_list[i]:.2f},ssim_phs:{ssim_phs_list[i]:.2f}')
#         img_diff_tensor, label_tensor = dataset_test[i]
#         pred, y_rec = model(img_diff_tensor.unsqueeze(0),TF_torch)
#         pred = pred.squeeze(0)
#         tensor_numpy_show(img_diff_tensor,label_tensor)
#         tensor_numpy_show(y_rec,pred)
    

In [ ]:
# from model import PINet_cpx_v6_pre
# model = PINet_cpx_v6_pre(fold_iters=fold_iters).to(device)
# model_path = f'model_pinet_size128_1.5mm_epoch200_batchsize16.pth'
# checkpoint_path = os.path.join(model_folder,model_path)
# load_checkpoint(checkpoint_path, model)

# import time
# psnr_amp_list = []
# psnr_phs_list = []
# ssim_amp_list = []
# ssim_phs_list = []
# time_list = []
# count = 0

# for img_diff_tensor, label_tensor in dataset_test:
#     count += 1
#     label_tensor = label_tensor.to(device)
#     start_time = time.time()
#     pred, y_rec = model(img_diff_tensor.unsqueeze(0),TF_torch)
#     pred = pred.squeeze(0)
#     end_time = time.time()
#     time_used = end_time - start_time
#     time_list.append(time_used)
#     psnr_amp = compute_psnr_skimage_single(torch.abs(pred),torch.abs(label_tensor))
#     psnr_phs = compute_psnr_skimage_single(torch.angle(pred),torch.angle(label_tensor))
#     ssim_amp = compute_ssim_skimage_single(torch.abs(pred),torch.abs(label_tensor))
#     ssim_phs = compute_ssim_skimage_single(torch.angle(pred),torch.angle(label_tensor))
#     psnr_amp_list.append(psnr_amp)
#     psnr_phs_list.append(psnr_phs)
#     ssim_amp_list.append(ssim_amp)
#     ssim_phs_list.append(ssim_phs)
#     print(f'image {count}: psnr_amp: psnr_amp:{psnr_amp:.2f}dB,psnr_phs:{psnr_phs:.2f}dB,ssim_amp:{ssim_amp:.2f},ssim_phs:{ssim_phs:.2f}, time:{time_used:.2f}s')

# psnr_amp_avg = np.mean(psnr_amp_list)
# psnr_phs_avg = np.mean(psnr_phs_list)
# ssim_amp_avg = np.mean(ssim_amp_list)
# ssim_phs_avg = np.mean(ssim_phs_list)
# time_avg = np.mean(time_list)
# print(f"{model_path}: psnr_amp:{psnr_amp_avg:.2f}dB,psnr_phs:{psnr_phs_avg:.2f}dB,ssim_amp:{ssim_amp_avg:.2f},ssim_phs:{ssim_phs_avg:.2f},time_avg:{time_avg:.2f}s")  

In [ ]:
# n = 100      # 图片序号
# model = PINet_cpx_v6(fold_iters=fold_iters).to(device)
# model_path = f'model_pinet_ours_2mm_epoch200_batchsize16.pth'
# checkpoint_path = os.path.join(model_folder,model_path)
# load_checkpoint(checkpoint_path, model)

# img_diff_tensor, label_tensor = dataset_test[n-1]
# pred, y_rec = model(img_diff_tensor.unsqueeze(0),TF_torch)
# pred = pred.squeeze(0)
# y_rec = y_rec.squeeze(0)
# psnr_amp = compute_psnr_skimage_single(torch.abs(pred),torch.abs(label_tensor))
# psnr_phs = compute_psnr_skimage_single(torch.angle(pred),torch.angle(label_tensor))
# ssim_amp = compute_ssim_skimage_single(torch.abs(pred),torch.abs(label_tensor))
# ssim_phs = compute_ssim_skimage_single(torch.angle(pred),torch.angle(label_tensor))
# print(f'model_v6 : psnr_amp: psnr_amp:{psnr_amp:.2f}dB,psnr_phs:{psnr_phs:.2f}dB,ssim_amp:{ssim_amp:.2f},ssim_phs:{ssim_phs:.2f}')
# tensor_numpy_show(img_diff_tensor,label_tensor)
# tensor_numpy_show(y_rec,pred)

# save_folder = './results_exp'
# if not os.path.exists(save_folder):
#     os.makedirs(save_folder)
    
# save_path_amp = os.path.join(save_folder,f'amp_ours_{n}.png')
# save_path_phs = os.path.join(save_folder,f'phs_ours_{n}.png')
# save_path_diff = os.path.join(save_folder,f'diff_{n}.png')
# save_path_amp_original = os.path.join(save_folder,f'amp_original_{n}.png')
# save_path_phs_original = os.path.join(save_folder,f'phs_original_{n}.png')
# amp_np = torch.abs(pred).squeeze().detach().cpu().numpy()
# phs_np = torch.angle(pred).squeeze().detach().cpu().numpy()
# diff_np = img_diff_tensor.squeeze().detach().cpu().numpy()
# amp_np_original = torch.abs(label_tensor).squeeze().detach().cpu().numpy()
# phs_np_original = torch.angle(label_tensor).squeeze().detach().cpu().numpy()


# amp_np = np.uint8(255 * amp_np / np.max(amp_np))
# phs_np = np.uint8(255 * phs_np / np.max(phs_np))
# diff_np = np.uint8(255 * diff_np / np.max(diff_np))
# amp_np_original = np.uint8(255 * amp_np_original / np.max(amp_np_original))
# phs_np_original = np.uint8(255 * phs_np_original / np.max(phs_np_original))

# amp_img = Image.fromarray(amp_np, mode='L')
# phs_img = Image.fromarray(phs_np, mode='L')
# diff_img = Image.fromarray(diff_np, mode='L')
# amp_img_original = Image.fromarray(amp_np_original, mode='L')
# phs_img_original = Image.fromarray(phs_np_original, mode='L')

# amp_img.save(save_path_amp)
# phs_img.save(save_path_phs)
# diff_img.save(save_path_diff)
# amp_img_original.save(save_path_amp_original)
# phs_img_original.save(save_path_phs_original)

In [ ]:
# n = 2      # 图片序号
# model = PINet_cpx_v6(fold_iters=fold_iters).to(device)
# model_path = f'model_pinet_ours_2mm_epoch200_batchsize16.pth'
# checkpoint_path = os.path.join(model_folder,model_path)
# load_checkpoint(checkpoint_path, model)

# img_diff_tensor, label_tensor = dataset_test[n-1]
# pred, y_rec = model(img_diff_tensor.unsqueeze(0),TF_torch)
# pred = pred.squeeze(0)
# y_rec = y_rec.squeeze(0)
# psnr_amp = compute_psnr_skimage_single(torch.abs(pred),torch.abs(label_tensor))
# psnr_phs = compute_psnr_skimage_single(torch.angle(pred),torch.angle(label_tensor))
# ssim_amp = compute_ssim_skimage_single(torch.abs(pred),torch.abs(label_tensor))
# ssim_phs = compute_ssim_skimage_single(torch.angle(pred),torch.angle(label_tensor))
# print(f'model_v6 : psnr_amp: psnr_amp:{psnr_amp:.2f}dB,psnr_phs:{psnr_phs:.2f}dB,ssim_amp:{ssim_amp:.2f},ssim_phs:{ssim_phs:.2f}')
# tensor_numpy_show(img_diff_tensor,label_tensor)
# tensor_numpy_show(y_rec,pred)

# save_folder = './results_exp'
# if not os.path.exists(save_folder):
#     os.makedirs(save_folder)
    
# save_path_amp = os.path.join(save_folder,f'amp_ours_{n}.png')
# save_path_phs = os.path.join(save_folder,f'phs_ours_{n}.png')
# save_path_diff = os.path.join(save_folder,f'diff_{n}.png')
# save_path_amp_original = os.path.join(save_folder,f'amp_original_{n}.png')
# save_path_phs_original = os.path.join(save_folder,f'phs_original_{n}.png')
# amp_np = torch.abs(pred).squeeze().detach().cpu().numpy()
# phs_np = torch.angle(pred).squeeze().detach().cpu().numpy()
# diff_np = img_diff_tensor.squeeze().detach().cpu().numpy()
# amp_np_original = torch.abs(label_tensor).squeeze().detach().cpu().numpy()
# phs_np_original = torch.angle(label_tensor).squeeze().detach().cpu().numpy()


# amp_np = np.uint8(255 * amp_np / np.max(amp_np))
# phs_np = np.uint8(255 * phs_np / np.max(phs_np))
# diff_np = np.uint8(255 * diff_np / np.max(diff_np))
# amp_np_original = np.uint8(255 * amp_np_original / np.max(amp_np_original))
# phs_np_original = np.uint8(255 * phs_np_original / np.max(phs_np_original))

# amp_img = Image.fromarray(amp_np, mode='L')
# phs_img = Image.fromarray(phs_np, mode='L')
# diff_img = Image.fromarray(diff_np, mode='L')
# amp_img_original = Image.fromarray(amp_np_original, mode='L')
# phs_img_original = Image.fromarray(phs_np_original, mode='L')

# amp_img.save(save_path_amp)
# phs_img.save(save_path_phs)
# diff_img.save(save_path_diff)
# amp_img_original.save(save_path_amp_original)
# phs_img_original.save(save_path_phs_original)

In [ ]:
# def mse_loss(x1,x2):
#     # return ((x1-x2)**2).mean()
#     return torch.sum((x1-x2)**2)

# def get_lr(optimizer):
#     for param_group in optimizer.param_groups:
#         return param_group['lr']

# def change_lr(optimizer,lr):
#     for param_group in optimizer.param_groups:
#         param_group['lr'] = lr

In [ ]:
# from model import PINet_cpx_v1

In [ ]:

# # Neural Network Model
# model = PINet_cpx_v1(fold_iters=fold_iters).to(device)
# # Optimizer
# lr = 1e-4
# optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# n = 100
# img_diff_tensor, label_tensor = dataset_test[n-1]

# start_time = time.time()
# for i in range(1000):
    
#     model.train()
#     model.zero_grad()
#     pred, y_rec = model(img_diff_tensor.unsqueeze(0).to(device),TF_torch)
#     loss = mse_loss(y_rec.squeeze(0), img_diff_tensor)
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()
    
#     if((i+1) % 500 == 0):
#         psnr_amp = compute_psnr_skimage_single(torch.abs(pred.squeeze(0)),torch.abs(label_tensor))
#         psnr_phs = compute_psnr_skimage_single(torch.angle(pred.squeeze(0)),torch.angle(label_tensor))
#         ssim_amp = compute_ssim_skimage_single(torch.abs(pred.squeeze(0)),torch.abs(label_tensor))
#         ssim_phs = compute_ssim_skimage_single(torch.angle(pred.squeeze(0)),torch.angle(label_tensor))
        
#         # 将张量转换为 NumPy 数组并归一化到 [0, 1] 的范围
#         amp_np = torch.abs(pred).squeeze().detach().cpu().numpy()  # 去掉通道维度并转为 NumPy 数组
#         phs_np = torch.angle(pred).squeeze().detach().cpu().numpy()

#         # 线性归一化到 [0, 1]
#         amp_np = normalize_tensor(amp_np)
#         phs_np = normalize_tensor(phs_np)

#         fig, ax = plt.subplots(1, 2, figsize=(12, 6))
#         ax[0].imshow(amp_np, cmap='gray') 
#         ax[0].set_title('Amp')
#         ax[0].axis('off')  
#         ax[1].imshow(phs_np, cmap='gray')
#         ax[1].set_title('Phs')
#         ax[1].axis('off') 

#         plt.show()
        
#         print(f"model_unn_epoch {i+1}: psnr_amp:{psnr_amp:.2f}dB,psnr_phs:{psnr_phs:.2f}dB,ssim_amp:{ssim_amp:.2f},ssim_phs:{ssim_phs:.2f} ")  
# end_time = time.time()
# time_used = end_time - start_time
# print(f"time used: {time_used:.2f}s")
# save_path_amp = os.path.join(save_folder,f'amp_unn_{n}.png')
# save_path_phs = os.path.join(save_folder,f'phs_unn_{n}.png')
# save_path_diff = os.path.join(save_folder,f'diff_{n}.png')
# save_path_amp_original = os.path.join(save_folder,f'amp_original_{n}.png')
# save_path_phs_original = os.path.join(save_folder,f'phs_original_{n}.png')
# amp_np = torch.abs(pred).squeeze().detach().cpu().numpy()
# phs_np = torch.angle(pred).squeeze().detach().cpu().numpy()
# diff_np = img_diff_tensor.squeeze().detach().cpu().numpy()
# amp_np_original = torch.abs(label_tensor).squeeze().detach().cpu().numpy()
# phs_np_original = torch.angle(label_tensor).squeeze().detach().cpu().numpy()
# amp_np = (amp_np - np.min(amp_np))/ (np.max(amp_np) - np.min(amp_np))
# phs_np = (phs_np - np.min(phs_np))/ (np.max(phs_np) - np.min(phs_np))
# amp_np = np.uint8(255 * amp_np)
# phs_np = np.uint8(255 * phs_np)
# amp_img = Image.fromarray(amp_np, mode='L')
# phs_img = Image.fromarray(phs_np, mode='L')
# amp_img.save(save_path_amp)
# phs_img.save(save_path_phs)
# # diff_img.save(save_path_diff)
# # amp_img_original.save(save_path_amp_original)
# # phs_img_original.save(save_path_phs_original)

In [ ]:
# model = PINet_cpx_v6(fold_iters=fold_iters).to(device)
# lr = 1e-4
# optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# for i in range(500):
#     model.train()
#     model.zero_grad()
#     pred, y_rec = model(diff_tensor.unsqueeze(0).unsqueeze(0).to(device),TF_torch)
#     loss = mse_loss(y_rec.squeeze(0), diff_tensor.unsqueeze(0))
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()
#     if((i+1) % 50 == 0):
        
    
#         # 将张量转换为 NumPy 数组并归一化到 [0, 1] 的范围
#         amp_np = torch.abs(pred).squeeze().detach().cpu().numpy()  # 去掉通道维度并转为 NumPy 数组
#         phs_np = torch.angle(pred).squeeze().detach().cpu().numpy()

#         # 线性归一化到 [0, 1]
#         amp_np = normalize_tensor(amp_np)
#         phs_np = normalize_tensor(phs_np)

#         fig, ax = plt.subplots(1, 2, figsize=(12, 6))
#         ax[0].imshow(amp_np, cmap='gray') 
#         ax[0].set_title('Amp_{i+1}epochs')
#         ax[0].axis('off')  
#         ax[1].imshow(phs_np, cmap='gray')
#         ax[1].set_title('Phs_{i+1}epochs')
#         ax[1].axis('off') 

#         plt.show()
    